In [ ]:
!pip install -q "transformers>=5.0.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
from PIL import Image
# Import trực tiếp class của Paddle để tránh lỗi AutoMapping
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoConfig

os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

model_path = "PaddlePaddle/PaddleOCR-VL-1.5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Đang nạp PaddleOCR-VL 1.5 bằng phương thức chuyên biệt...")

if 'model' not in locals():
    # 1. Load config và ép kiểu thủ công để tránh lỗi text_config
    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)

    # Một số bản transformers mới yêu cầu fix cứng text_config nếu nó bị thiếu
    if not hasattr(config, 'text_config'):
        # Trỏ text_config về chính nó nếu cấu trúc model phẳng
        config.text_config = config

    # 2. Load model sử dụng class cụ thể hoặc AutoModel với config đã sửa
    model = AutoModelForImageTextToText.from_pretrained(
        model_path,
        config=config,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        # Sử dụng dtype thay cho torch_dtype nếu transformers yêu cầu
        dtype=torch.bfloat16
    ).to(DEVICE).eval()

    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    print("✓ Đã vượt qua lỗi Config. Model sẵn sàng!")
else:
    print("✓ Model đã có sẵn trong bộ nhớ.")

In [ ]:
import os
import json
import torch
from PIL import Image
from tqdm import tqdm

# --- CONFIG ---
folder_path = "/content/drive/Shareddrives/NLP4B/MAIN/data/keyframe_new/hoang/19hfQm5-3Mo"
output_dir = "/content/drive/Shareddrives/NLP4B/MAIN/data/ocr/hoang"
batch_size = 6  # Trên T4, 4-bit/bfloat16 có thể lên được 6-8
task = "ocr"

# Tối ưu hệ thống Torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True # Tối ưu thuật toán convolution cho phần cứng hiện tại

folder_name = os.path.basename(os.path.normpath(folder_path))
output_file = os.path.join(output_dir, f"{folder_name}.json")
os.makedirs(output_dir, exist_ok=True)

image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
final_results = []

print(f"🚀 Chế độ Turbo OCR: {len(image_files)} ảnh | Batch: {batch_size}")

# Sử dụng inference_mode để đạt tốc độ cao nhất
with torch.inference_mode():
    for i in tqdm(range(0, len(image_files), batch_size)):
        batch_filenames = image_files[i : i + batch_size]
        batch_images = []

        for fname in batch_filenames:
            img_path = os.path.join(folder_path, fname)
            try:
                # Mở ảnh và ép về size chuẩn để batch hóa nhanh hơn
                with Image.open(img_path) as img:
                    # Fix size giúp model tránh tính toán lại position embedding cho mỗi ảnh khác size
                    batch_images.append(img.convert("RGB").copy())
            except: continue

        if not batch_images: continue

        # Tiền xử lý tập trung
        max_pixels = 1280 * 28 * 28 # Mức tiêu chuẩn cho OCR nhanh
        batch_messages = [[{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": "OCR:"}]}] for img in batch_images]

        inputs = processor.apply_chat_template(
            batch_messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            # Giữ shortest_edge cố định giúp tối ưu hóa bộ nhớ đệm GPU
            images_kwargs={"size": {"shortest_edge": 448, "longest_edge": max_pixels}},
        ).to(model.device, dtype=torch.bfloat16) # Ép kiểu dữ liệu tensor ngay khi nạp vào GPU

        # Generate với các tham số tối ưu (tắt các tính năng search phức tạp)
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,      # Tắt lấy mẫu ngẫu nhiên để chạy nhanh hơn (Greedy search)
            use_cache=True,       # Quan trọng: Dùng lại các key/value đã tính toán
            pad_token_id=processor.tokenizer.pad_token_id
        )

        # Hậu xử lý (Post-processing)
        prompt_len = inputs["input_ids"].shape[-1]
        for idx, output_tensor in enumerate(outputs):
            text = processor.decode(output_tensor[prompt_len:-1], skip_special_tokens=True)
            final_results.append({"file_name": batch_filenames[idx], "ocr_result": text.strip()})

        # Giải phóng bộ nhớ nhanh
        del inputs, outputs
        if i % 20 == 0: # Cứ sau 20 batch thì dọn dẹp cache một lần để tránh treo
            torch.cuda.empty_cache()

# Lưu kết quả
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=4)